# clean_taxi_fhv

Pulls 2025 Yellow Taxi and High Volume FHV trips from the NYC Open Data API, then cleans them. The full tables are too big to page through, so the notebook samples: every `STRIDE`-th day of 2025, one request per hour, at most `ROWS_PER_SLICE` trips per hour. Raise those numbers for more data.

Per trip it keeps pickup/dropoff time, taxi zones, distance and fare, and adds `minutes`, `mph`, `start_hour` and `dow`.

Rows are dropped when they are missing a value or fail any of these checks:
- pickup outside 2025
- duration under 1 or over 180 minutes
- distance of 0 or over 100 miles
- pickup or dropoff zone not in 1-263
- fare of 0 or less
- average speed under 0.5 or over 80 mph

In [ ]:
import time
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor
from functools import reduce
from pathlib import Path

import pandas as pd
import requests

APP_TOKEN = "" 
YELLOW_ID = "irx2-vgm4"
FHV_ID    = "nt46-nv3c"

START, END = "2025-01-01", "2026-01-01"
STRIDE = 3                # pull every 3rd day, 3 cycles through all weekdays; 7 would not
ROWS_PER_SLICE = 1000     # max trips per hour per day
WORKERS = 8               # parallel requests
MIN_MIN, MAX_MIN = 1, 180

DATA_DIR = Path("/data")
BASE = "https://data.cityofnewyork.us/resource"
HEADERS = {"X-App-Token": APP_TOKEN} if APP_TOKEN else {}

## Helper Functions

In [ ]:
def get_json(dataset_id, params):
    for attempt in range(6):
        r = requests.get(f"{BASE}/{dataset_id}.json", params=params, headers=HEADERS, timeout=120)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503, 504):   # throttled or server hiccup: back off, retry
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()
    raise RuntimeError(f"gave up on {dataset_id} {params}")


def fetch_day(dataset_id, time_col, cols, day):
    """One request per hour of `day`, each capped at ROWS_PER_SLICE rows."""
    def one_hour(h):
        t0 = day + timedelta(hours=h)
        t1 = t0 + timedelta(hours=1)
        where = f"{time_col} >= '{t0:%Y-%m-%dT%H:%M:%S}' AND {time_col} < '{t1:%Y-%m-%dT%H:%M:%S}'"
        return get_json(dataset_id, {"$select": ",".join(cols), "$where": where, "$limit": ROWS_PER_SLICE})
    with ThreadPoolExecutor(WORKERS) as ex:
        hours = list(ex.map(one_hour, range(24)))
    return [tuple(r.get(c) for c in cols) for rows in hours for r in rows]


def sample_days():
    d, end = datetime.fromisoformat(START), datetime.fromisoformat(END)
    while d < end:
        yield d
        d += timedelta(days=STRIDE)


def pull_days(dataset_id, time_col, cols, folder):
    """Save each sampled day as its own CSV file. Days already saved are skipped."""
    folder.mkdir(parents=True, exist_ok=True)
    for day in sample_days():
        f = folder / f"{day:%Y-%m-%d}.csv"
        if f.exists():
            continue
        rows = fetch_day(dataset_id, time_col, list(cols), day)
        pd.DataFrame(rows, columns=list(cols)).to_csv(f, index=False)
        print(f"{day:%Y-%m-%d}: {len(rows):,} rows")


def load_raw(folder, cols):
    parts = [pd.read_csv(f, dtype=str) for f in sorted(folder.glob("*.csv"))]
    df = pd.concat([p for p in parts if len(p)], ignore_index=True)
    for c, t in cols.items():
        if t == "datetime":
            df[c] = pd.to_datetime(df[c], errors="coerce")
        elif t in ("int", "double"):
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def clean_trips(df, cols, dist_col, fare_col, name):
    """Apply the checks, print how many rows fail each one, return the clean rows."""
    d = df.copy()
    d["minutes"] = (d["dropoff_datetime"] - d["pickup_datetime"]).dt.total_seconds() / 60
    d["mph"] = d[dist_col] / (d["minutes"] / 60)

    rules = {
        "date_range": (d["pickup_datetime"] >= pd.Timestamp(START)) & (d["pickup_datetime"] < pd.Timestamp(END)),
        "duration":   d["minutes"].between(MIN_MIN, MAX_MIN),
        "distance":   (d[dist_col] > 0) & (d[dist_col] <= 100),
        "zones":      d["pulocationid"].between(1, 263) & d["dolocationid"].between(1, 263),
        "fare":       d[fare_col] > 0,
        "speed":      d["mph"].between(0.5, 80),
    }
    
    # missing values compare as False, so they count as failures
    ok = {k: v.fillna(False).astype(bool) for k, v in rules.items()}

    print(f"{name}: {len(d):,} rows")
    for k, v in ok.items():
        print(f"  fails {k}: {(~v).sum():,}")

    d = d[reduce(lambda a, b: a & b, ok.values())].copy()
    for c, t in cols.items():
        if t == "int":
            d[c] = d[c].astype("Int64")
    d["start_hour"] = d["pickup_datetime"].dt.hour
    d["dow"] = (d["pickup_datetime"].dt.dayofweek + 1) % 7 + 1   # 1 = Sunday ... 7 = Saturday
    return d.reset_index(drop=True)

## Yellow Taxi

In [ ]:
yellow_cols = {
    "vendorid": "int",
    "tpep_pickup_datetime": "datetime",
    "tpep_dropoff_datetime": "datetime",
    "passenger_count": "int",
    "trip_distance": "double",
    "pulocationid": "int",
    "dolocationid": "int",
    "fare_amount": "double",
    "total_amount": "double",
}
pull_days(YELLOW_ID, "tpep_pickup_datetime", yellow_cols, DATA_DIR / "raw_yellow")

In [ ]:
yellow = (load_raw(DATA_DIR / "raw_yellow", yellow_cols)
          .rename(columns={"tpep_pickup_datetime": "pickup_datetime", "tpep_dropoff_datetime": "dropoff_datetime"}))
yellow_types = {("pickup_datetime" if k == "tpep_pickup_datetime" else "dropoff_datetime" if k == "tpep_dropoff_datetime" else k): v
                for k, v in yellow_cols.items()}
yellow_clean = clean_trips(yellow, yellow_types, "trip_distance", "fare_amount", "yellow")
yellow_clean.to_csv(DATA_DIR / "yellow_clean.csv", index=False)
print("saved", f"{len(yellow_clean):,}")

## High Volume HFV

In [ ]:
fhv_cols = {
    "hvfhs_license_num": "string",
    "pickup_datetime": "datetime",
    "dropoff_datetime": "datetime",
    "pulocationid": "int",
    "dolocationid": "int",
    "trip_miles": "double",
    "base_passenger_fare": "double",
    "shared_match_flag": "string",
}
pull_days(FHV_ID, "pickup_datetime", fhv_cols, DATA_DIR / "raw_fhv")

In [ ]:
fhv = load_raw(DATA_DIR / "raw_fhv", fhv_cols)
fhv_clean = clean_trips(fhv, fhv_cols, "trip_miles", "base_passenger_fare", "fhv")
fhv_clean.to_csv(DATA_DIR / "fhv_clean.csv", index=False)
print("saved", f"{len(fhv_clean):,}")

In [ ]:
for t, dist in [("yellow_clean", "trip_distance"), ("fhv_clean", "trip_miles")]:
    print(t)
    print(pd.read_csv(DATA_DIR / f"{t}.csv")[["minutes", "mph", dist]].describe().round(2))